# Pvlib Generation Process

This Jupyter notebook provides a brief overview of how to use the embedded **pvlib** functionality 
within the  **geodata** package to create photovoltaic energy series using **geodata** cutouts.

*The following guide assumes you have installed and configured **geodata** and all required dependencies, 
and have also installed the **pvlib** module.*

For a guide on how to install **pvlib**, see the [pvlib documentation](https://pvlib-python.readthedocs.io/en/stable/user_guide/installation.html#editablelibrary).


## Step 1 - Download and Create an ERA5 Cutout

Assuming you have previously created a CDS account and set up the CDS API credentials, you can download ERA5 data from the CDS API as follows.

First, define a dataset object for the data you wish to download:

In [ ]:
import geodata

DS = geodata.Dataset(
    module="era5",
    weather_data_config="wind_solar_hourly",
    years=slice(2009, 2009),
    months=slice(5, 6),
    bounds=[50, -3, 45, 3]
)

if DS.prepared == False:
    DS.get_data()

cutout = geodata.Cutout(
    name="era5-europe-test-2009-56",
    module="era5",
    weather_data_config="wind_solar_hourly",
    ys=slice(1, 2),   
    xs=slice(47, 48), 
    years=slice(2009, 2009),
    months=slice(5, 6),
)
cutout.prepare(overwrite=True)

* Specify "era5" as the model, with a `weather_data_config` of "wind_solar_hourly" (this is currently the only configuration supported by **pvlib** functionality in **geodata**.
* PV data will be generated at the hourly level for each set of coordinates included in the specified bounds, over the specified time period.

## Step 2 - Define PV system and model parameters

In order to generate PV data, you must first define parameters for both the PV system and the PV model.
First, define system parameters as per the following example:

In [ ]:
n_mods = 50
n_strings = 1
cec_modules = geodata.pvlib.retrieve_sam('CECMod')
module = cec_modules['Kaneka_U_SA105']
inv =  geodata.pvlib.retrieve_sam("CECInverter")['Fronius_USA__CL_33_3_Delta__208V_']

system = geodata.pvlib.pv_system(
    arrays = None,
    surface_tilt=35,
    surface_azimuth=180,
    racking_model = 'open_rack',
    module_parameters=module,
    modules_per_string = n_mods,
    module_type = 'glass_polymer',
    module = 'Kaneka_U_SA105',
    strings_per_inverter = n_strings, 
    inverter_parameters=inv
)

For a full list of module and inverter info available for use, see: [pvlib.pvsystem.retrieve_sam()](https://pvlib-python.readthedocs.io/en/stable/reference/generated/pvlib.pvsystem.retrieve_sam.html#pvlib.pvsystem.retrieve_sam)

For comprehensive documentation on parameters used to define a PV system with **pvlib**, see: [pvlib.pvsystem.PVSystem()](https://pvlib-python.readthedocs.io/en/stable/reference/generated/pvlib.pvsystem.PVSystem.html)

Next, define the PV model parameters as follows:

In [5]:
model_config = geodata.pvlib.ModelChainConfig(
    clearsky_model= 'haurwitz',
    transposition_model='perez', 
    solar_position_method= 'nrel_numpy',
    airmass_model= 'kastenyoung1989',
    dc_model='cec',
    ac_model='sandia', 
    aoi_model="physical",
    spectral_model='first_solar',
    dc_ohmic_model='no_loss'
)

For a comprehensive documentation on parameters used to define a PV model with **pvlib**, see: [pvlib.modelchain.ModelChain](https://pvlib-python.readthedocs.io/en/stable/reference/generated/pvlib.modelchain.ModelChain.html#pvlib.modelchain.ModelChain)

## Step 3 - Run PV model using cutout

The model can then be run as follows:

In [ ]:
model = geodata.pvlib.pvlib_model(
    cutout,
    system,
    model_config
)
model

If needed, the results can be converted to a Pandas dataframe, and/or graphed:

In [ ]:
model_df = model.to_dataframe()
print(model_df)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
x_coord = 47.0
y_coord = 1.25
date = '2009-05-02'
plot_df = model_df.reset_index()
plot_df = plot_df[
    (plot_df['x'] == x_coord) 
    & (plot_df['y'] == y_coord) 
    & (plot_df['time'].dt.date == pd.to_datetime(date).date())]

plot_df = plot_df.sort_values(by='time')
plt.figure(figsize=(10, 5))
plt.plot(plot_df['time'], plot_df['ac'], marker='o', linestyle='-')
plt.xlabel('Time')
plt.ylabel('AC Power Output')
plt.title(f'AC power output for ({x_coord}, {y_coord} on {date})')
plt.xticks(rotation=45)
plt.grid()
plt.show()